# 3. Retrieving cards: queries, pools, and the fidelity split

The last card-layer notebook: how you actually get cards back out, and how a
search decides which of them it is allowed to use.

In [1]:
from pathlib import Path

import duckdb

REPO = Path.cwd()
while not (REPO / "data" / "cards.duckdb").exists() and REPO != REPO.parent:
    REPO = REPO.parent

DB = REPO / "data" / "cards.duckdb"
con = duckdb.connect(str(DB), read_only=True)


def q(sql, params=None, limit=12):
    # Run a query and print it as a small table.
    rows = con.execute(sql, params or []).fetchall()
    names = [d[0] for d in con.description]
    shown = rows[:limit]
    if not shown:
        print("  ".join(map(str, names)))
        print("(no rows)")
        return rows
    widths = [max([len(str(n))] + [len(str(r[i])[:38]) for r in shown])
              for i, n in enumerate(names)]
    print("  ".join(str(n).ljust(w) for n, w in zip(names, widths)))
    print("  ".join("-" * w for w in widths))
    for row in rows[:limit]:
        print("  ".join(str(v)[:38].ljust(w) for v, w in zip(row, widths)))
    if len(rows) > limit:
        print(f"... {len(rows) - limit} more rows")
    return rows

In [2]:
from manalab.cards.store import CardStore

store = CardStore(DB)
print(f"{store.count():,} cards")

35,142 cards


## The store API

`CardStore` hides where card data came from. Swapping Scryfall for MTGJSON
changes one module and nothing above it.

In [3]:
card = store.by_name("Vampire Nighthawk")
print(f"by_name        {card.name}  {card.cost}  {card.power}/{card.toughness}")

resolved = store.resolve(["Forest", "Llanowar Elves", "Leatherback Baloth"])
print(f"resolve        {list(resolved)}")

sample = store.sample(4, seed=7)
print(f"sample(seed=7) {[c.name for c in sample]}")
print(f"sample(seed=7) {[c.name for c in store.sample(4, seed=7)]}   <- reproducible")

print(f"legal_in       {len(store.legal_in('modern')):,} Modern-legal card names")

by_name        Vampire Nighthawk  {1}{B}{B}  2/3
resolve        ['Forest', 'Llanowar Elves', 'Leatherback Baloth']
sample(seed=7) ['Sparring Regimen', 'Welcome to Sweettooth', 'Minecart Daredevil // Ride the Rails', 'Zephyr Boots']


sample(seed=7) ['Sparring Regimen', 'Welcome to Sweettooth', 'Minecart Daredevil // Ride the Rails', 'Zephyr Boots']   <- reproducible
legal_in       22,450 Modern-legal card names


## Querying directly

For anything the store does not expose, the database is right there. Colour
identity is the filter that matters most for this project.

In [4]:
q('''
select name, mana_cost, power, toughness, keywords
from cards
where color_identity = '["G"]'
  and json_extract_string(legalities, '$.modern') = 'legal'
  and type_line like '%Creature%'
  and cmc = 3
  and try_cast(power as integer) >= 4
order by try_cast(power as integer) desc, name
''', limit=10)

name                                mana_cost      power  toughness  keywords                        
----------------------------------  -------------  -----  ---------  --------------------------------
Force of Savagery                   {2}{G}         8      0          ['Trample']                     
Groundbreaker                       {G}{G}{G}      6      1          ['Haste', 'Trample']            
Pugnacious Hammerskull              {2}{G}         6      6          []                              
Ravaging Riftwurm                   {1}{G}{G}      6      6          ['Vanishing', 'Kicker']         
Fear of Exposure                    {2}{G}         5      4          ['Trample']                     
Lovestruck Beast // Heart's Desire  {2}{G} // {G}  5      5          []                              
Regal Imperiosaur                   {1}{G}{G}      5      4          []                              
Rhonas the Indomitable              {2}{G}         5      5          ['Indestructi

[('Force of Savagery', '{2}{G}', '8', '0', ['Trample']),
 ('Groundbreaker', '{G}{G}{G}', '6', '1', ['Haste', 'Trample']),
 ('Pugnacious Hammerskull', '{2}{G}', '6', '6', []),
 ('Ravaging Riftwurm', '{1}{G}{G}', '6', '6', ['Vanishing', 'Kicker']),
 ('Fear of Exposure', '{2}{G}', '5', '4', ['Trample']),
 ("Lovestruck Beast // Heart's Desire", '{2}{G} // {G}', '5', '5', []),
 ('Regal Imperiosaur', '{1}{G}{G}', '5', '4', []),
 ('Rhonas the Indomitable',
  '{2}{G}',
  '5',
  '5',
  ['Indestructible', 'Deathtouch']),
 ('Steel Leaf Champion', '{G}{G}{G}', '5', '4', []),
 ('Wayward Swordtooth', '{2}{G}', '5', '5', ['Ascend']),
 ('Afterburner Expert', '{2}{G}', '4', '2', ['Exhaust']),
 ('Ainok Artillerist', '{2}{G}', '4', '1', []),
 ('Alpine Grizzly', '{2}{G}', '4', '2', []),
 ('Ambush Wolf', '{2}{G}', '4', '2', ['Flash']),
 ('Bloated Contaminator',
  '{2}{G}',
  '4',
  '4',
  ['Toxic', 'Trample', 'Proliferate']),
 ('Boon Satyr', '{1}{G}{G}', '4', '2', ['Bestow', 'Flash']),
 ('Burrowing Razorma

### Colour identity is not colour

A trap worth internalising. **Colour** is the mana symbols in the cost.
**Colour identity** adds every coloured symbol in the rules text — which is why
Noble Hierarch is not a mono-green card despite costing `{G}`.

In [5]:
q('''
select name, mana_cost, colors, color_identity, oracle_text
from cards
where name in ('Noble Hierarch', 'Birds of Paradise', 'Llanowar Elves',
               'Elves of Deep Shadow', 'Avacyn''s Pilgrim')
''')

name                  mana_cost  colors  color_identity   oracle_text                           
--------------------  ---------  ------  ---------------  --------------------------------------
Avacyn's Pilgrim      {G}        ['G']   ['G', 'W']       {T}: Add {W}.                         
Elves of Deep Shadow  {G}        ['G']   ['B', 'G']       {T}: Add {B}. This creature deals 1 da
Llanowar Elves        {G}        ['G']   ['G']            {T}: Add {G}.                         
Noble Hierarch        {G}        ['G']   ['G', 'U', 'W']  Exalted (Whenever a creature you contr
Birds of Paradise     {G}        ['G']   ['G']            Flying
{T}: Add one mana of any color.


[("Avacyn's Pilgrim", '{G}', ['G'], ['G', 'W'], '{T}: Add {W}.'),
 ('Elves of Deep Shadow',
  '{G}',
  ['G'],
  ['B', 'G'],
  '{T}: Add {B}. This creature deals 1 damage to you.'),
 ('Llanowar Elves', '{G}', ['G'], ['G'], '{T}: Add {G}.'),
 ('Noble Hierarch',
  '{G}',
  ['G'],
  ['G', 'U', 'W'],
  'Exalted (Whenever a creature you control attacks alone, that creature gets +1/+1 until end of turn.)\n{T}: Add {G}, {W}, or {U}.'),
 ('Birds of Paradise',
  '{G}',
  ['G'],
  ['G'],
  'Flying\n{T}: Add one mana of any color.')]

In [6]:
print("Birds of Paradise adds 'one mana of any color' — no coloured symbol,")
print("so its identity stays mono-green. Noble Hierarch names {W} and {U},")
print("so it does not belong in a mono-green deck at all.\n")
for name in ["Birds of Paradise", "Noble Hierarch"]:
    card = store.by_name(name)
    print(f"  {name:<20} colors={sorted(card.colors)}  "
          f"identity={sorted(card.color_identity)}")

Birds of Paradise adds 'one mana of any color' — no coloured symbol,
so its identity stays mono-green. Noble Hierarch names {W} and {U},
so it does not belong in a mono-green deck at all.

  Birds of Paradise    colors=['G']  identity=['G']
  Noble Hierarch       colors=['G']  identity=['G', 'U', 'W']


## Tags: what a card *does*, beside what it is

`card_tags` carries Scryfall's oracle tags — a human-curated functional
classification. Useful for questions the type line cannot answer, like "which
cards are removal".

In [7]:
q('''
select t.slug, t.label, t.card_count
from tags t
order by t.card_count desc
''', limit=15)

slug                           label                          card_count
-----------------------------  -----------------------------  ----------
activated-ability              activated ability              9099      
triggered-ability              triggered ability              7890      
spot-removal                   spot removal                   5026      
single-target-instant-sorcery  single target instant/sorcery  4676      
evasion                        evasion                        4609      
alliteration                   alliteration                   4418      
repeatable-crime               repeatable crime               3704      
unique-type-line               unique type line               2197      
intervening-if-clause          intervening if clause          2194      
attack-trigger                 attack trigger                 2065      
removal-creature               removal-creature               1935      
repeatable-removal             repeatable removal  

[('activated-ability', 'activated ability', 9099),
 ('triggered-ability', 'triggered ability', 7890),
 ('spot-removal', 'spot removal', 5026),
 ('single-target-instant-sorcery', 'single target instant/sorcery', 4676),
 ('evasion', 'evasion', 4609),
 ('alliteration', 'alliteration', 4418),
 ('repeatable-crime', 'repeatable crime', 3704),
 ('unique-type-line', 'unique type line', 2197),
 ('intervening-if-clause', 'intervening if clause', 2194),
 ('attack-trigger', 'attack trigger', 2065),
 ('removal-creature', 'removal-creature', 1935),
 ('repeatable-removal', 'repeatable removal', 1778),
 ('gains-pp-counters', 'gains pp counters', 1764),
 ('removal-destroy', 'removal-destroy', 1720),
 ('repeatable-pp-counters', 'repeatable pp counters', 1717),
 ('noncreature-typal', 'noncreature typal', 1659),
 ('namesake-spell', 'namesake spell', 1633),
 ('attacking-matters-self', 'attacking matters-self', 1619),
 ('virtual-vanilla', 'virtual vanilla', 1605),
 ('draw-engine', 'draw engine', 1565),
 ('f

In [8]:
q('''
select c.name, c.mana_cost, c.type_line
from card_tags ct
join cards c using (oracle_id)
where ct.tag = 'spot-removal'
  and c.color_identity = '["R"]'
  and json_extract_string(c.legalities, '$.modern') = 'legal'
  and c.cmc <= 1
order by c.name
''', limit=10)

name                    mana_cost  type_line                          
----------------------  ---------  -----------------------------------
Banefire                {X}{R}     Sorcery                            
Barbarian Ring                     Land                               
Barrage of Expendables  {R}        Enchantment                        
Blaze                   {X}{R}     Sorcery                            
Blazing Bomb            {R}        Creature — Elemental               
Blighted Gorge                     Land                               
Burn from Within        {X}{R}     Sorcery                            
Burst Lightning         {R}        Instant                            
By Force                {X}{R}     Sorcery                            
Cacophony Scamp         {R}        Creature — Phyrexian Goblin Warrior
... 124 more rows


[('Banefire', '{X}{R}', 'Sorcery'),
 ('Barbarian Ring', '', 'Land'),
 ('Barrage of Expendables', '{R}', 'Enchantment'),
 ('Blaze', '{X}{R}', 'Sorcery'),
 ('Blazing Bomb', '{R}', 'Creature — Elemental'),
 ('Blighted Gorge', '', 'Land'),
 ('Burn from Within', '{X}{R}', 'Sorcery'),
 ('Burst Lightning', '{R}', 'Instant'),
 ('By Force', '{X}{R}', 'Sorcery'),
 ('Cacophony Scamp', '{R}', 'Creature — Phyrexian Goblin Warrior'),
 ("Chandra's Defeat", '{R}', 'Instant'),
 ('Channeled Dragonfire', '{R}', 'Sorcery'),
 ('Clear a Path', '{R}', 'Sorcery'),
 ('Collateral Damage', '{R}', 'Instant'),
 ('Conflagrate', '{X}{X}{R}', 'Sorcery'),
 ('Crack the Earth', '{R}', 'Sorcery — Arcane'),
 ("Crater's Claws", '{X}{R}', 'Sorcery'),
 ('Crush', '{R}', 'Instant'),
 ('Deface', '{R}', 'Sorcery'),
 ('Demonfire', '{X}{R}', 'Sorcery'),
 ('Detonate', '{X}{R}', 'Sorcery'),
 ("Devil's Play", '{X}{R}', 'Sorcery'),
 ('Disintegrate', '{X}{R}', 'Sorcery'),
 ("Dreadmaw's Ire", '{R}', 'Instant'),
 ('Dual Shot', '{R}', 'In

## From cards to a searchable pool

`CandidatePool` is the entire interface a deck search needs: a fixed axis order,
a per-card maximum, and a target size. A decklist becomes an integer vector, and
that is what a genetic algorithm mutates.

In [9]:
from manalab.evaluate import pool as P

pool = P.mono_color(store, "G", fmt="modern")
print(f"faithful pool : {pool.size:>5} cards   <- the search space")
print(f"text tier     : {len(pool.text_tier):>5} cards   <- playable, text ignored")
print(f"\nfirst axes: {pool.pool.card_ids[:6]}")
print(f"upper bounds: {pool.pool.upper_bounds[:6]}  (basics are capped at deck size)")
print(f"deck size: {pool.pool.deck_size}")

faithful pool :   187 cards   <- the search space
text tier     :  3009 cards   <- playable, text ignored

first axes: ('Dryad Arbor', 'Forest', 'Snow-Covered Forest', 'Ankle Biter', 'Boreal Druid', 'Charging Badger')
upper bounds: (4, 60, 4, 4, 4, 4)  (basics are capped at deck size)
deck size: 60


In [10]:
from manalab.cards import decklist as DL

deck = DL.load(REPO / "decks" / "mono-green-ga.txt")
vector = pool.pool.to_vector(deck)

print(f"the evolved deck as a vector of length {len(vector)}:")
print(f"  nonzero entries: {sum(1 for v in vector if v)}")
print(f"  sums to: {sum(vector)}")
print(f"  legal:   {pool.pool.is_legal(deck)}")
print()
back = pool.pool.from_vector(vector)
print(f"round-trips exactly: {back.counts == deck.counts}")

the evolved deck as a vector of length 187:
  nonzero entries: 21
  sums to: 60
  legal:   True

round-trips exactly: True


## The fidelity split

The pool is divided by whether the engine can be *trusted* on a card, not by
whether the card is good. Optimising over cards whose text is ignored produces a
confident, wrong answer.

In [11]:
from manalab.engine import fidelity

print("in the searchable pool:")
for name in ["Leatherback Baloth", "Llanowar Elves", "Lightning Bolt"]:
    try:
        card = store.by_name(name)
    except Exception:
        continue
    report = fidelity.assess(card)
    print(f"  {'OK ' if report.faithful else 'NO '} {name:<22} "
          f"{'; '.join(report.reasons) or 'every word modelled'}")

print("\nset aside, with the reason:")
for card in list(pool.text_tier)[:6]:
    print(f"  NO  {card.name:<22} {'; '.join(pool.why(card.name))}")

in the searchable pool:
  OK  Leatherback Baloth     every word modelled
  OK  Llanowar Elves         every word modelled
  OK  Lightning Bolt         every word modelled

set aside, with the reason:
  NO  Argoth, Sanctum of Nature multi-faced; the engine plays only one side
  NO  Avabruck Caretaker // Hollowhenge Huntmaster multi-faced; the engine plays only one side
  NO  Azusa's Many Journeys // Likeness of the Seeker multi-faced; the engine plays only one side
  NO  Ba Sing Se             land with text the engine ignores
  NO  Bala Ged Recovery // Bala Ged Sanctuary multi-faced; the engine plays only one side
  NO  Bird Admirer // Wing Shredder multi-faced; the engine plays only one side


In [12]:
# The cards you would most want are mostly in the text tier. That is the
# honest scope statement for any result the search produces.
wanted = ["Tarmogoyf", "Craterhoof Behemoth", "Steel Leaf Champion",
          "Primeval Titan", "Nullhide Ferox", "Scavenging Ooze"]
for name in wanted:
    card = store.by_name(name)
    report = fidelity.assess(card)
    where = "SEARCHABLE" if report.faithful else "text tier"
    print(f"  {name:<22} {where:<12} {'; '.join(report.reasons)}")

  Tarmogoyf              text tier    characteristic-defining power/toughness (a '*'); rules text the engine does not execute
  Craterhoof Behemoth    text tier    rules text the engine does not execute
  Steel Leaf Champion    text tier    rules text the engine does not execute
  Primeval Titan         text tier    rules text the engine does not execute
  Nullhide Ferox         text tier    unimplemented keywords: hexproof; rules text the engine does not execute
  Scavenging Ooze        text tier    rules text the engine does not execute


## Building a gauntlet from a pool

The last step before a search can run: fixed opponents, assembled only from cards
the engine models exactly, so a loss is never the engine failing to read an
opponent's card.

In [13]:
import collections

from manalab.evaluate import gauntlet as G
from manalab.evaluate.match import gauntlet_fidelity

gauntlet = G.standard(pool)
print(f"blind spots in the gauntlet: {gauntlet_fidelity(gauntlet) or 'none'}\n")
for deck in gauntlet:
    counts = collections.Counter(c.name for c in deck.cards)
    lands = counts.get("Forest", 0)
    top = ", ".join(f"{n}x {name}" for name, n in counts.most_common(3)
                    if name != "Forest")
    print(f"  {deck.name:<10} {lands:>2} lands | {top}")

blind spots in the gauntlet: none

  aggro      20 lands | 4x Alpine Grizzly, 4x Gnottvold Recluse
  midrange   24 lands | 4x Gigantosaurus, 4x Kalonian Tusker
  fatties    26 lands | 4x Quakestrider Ceratops, 4x Gigantosaurus
  defensive  24 lands | 4x Gigantosaurus, 4x Pheres-Band Centaurs
  evasion    22 lands | 4x Elfhame Wurm, 4x Stomper Cub


## The whole path, end to end

    Scryfall bulk export
        -> cards / faces / face_types            (notebook 1)
        -> Effect(verb, params) per face         (notebook 2)
        -> fidelity: can the engine execute it?  (engine notebook 4)
        -> CandidatePool: the faithful tier      (this notebook)
        -> MatchEvaluator: decklist -> win rate
        -> GeneticSearch: mutate the vector

Each arrow is a boundary, and each boundary is where a wrong answer would come
from. The one worth re-reading is the fidelity step: it is the only thing
standing between a search and a confident conclusion about cards the engine
cannot read.